# 01-01 异步编程 async/await

**为什么 Agent 开发必须会异步编程？**

在 AI Agent 开发中，异步编程无处不在：
- 并发调用多个 LLM API（同时向 GPT-4 和 Claude 发请求）
- 同时执行多个工具（搜索 + 计算 + 数据库查询）
- 流式输出（streaming）处理
- 多 Agent 并行执行

**本节目标**：
- 理解 Python 异步模型（event loop, coroutine, task）
- 掌握 `async/await` 语法
- 学会用 `asyncio.gather` 并发执行任务
- 实战：并发调用 3 个 LLM API

---

## 1. 同步 vs 异步：为什么需要异步？

想象你要调用 3 个 LLM API，每个大约 2 秒：
- **同步**: 顺序等待 → 6 秒
- **异步**: 并发等待 → ~2 秒

对于 I/O 密集型任务（网络请求、文件读写），异步编程能极大提升吞吐量。

In [ ]:
import asyncio
import time

# ---- 同步版本 ----
def sync_task(name: str, delay: float) -> str:
    time.sleep(delay)  # 模拟网络等待
    return f"{name} 完成"

start = time.time()
results = [
    sync_task("LLM-A", 1.0),
    sync_task("LLM-B", 1.0),
    sync_task("LLM-C", 1.0),
]
print(f"同步耗时: {time.time() - start:.2f}s")
print(results)

In [ ]:
# ---- 异步版本 ----
async def async_task(name: str, delay: float) -> str:
    await asyncio.sleep(delay)  # 非阻塞等待
    return f"{name} 完成"

async def main():
    start = time.time()
    # gather 并发执行所有任务
    results = await asyncio.gather(
        async_task("LLM-A", 1.0),
        async_task("LLM-B", 1.0),
        async_task("LLM-C", 1.0),
    )
    print(f"异步耗时: {time.time() - start:.2f}s")
    print(results)

await main()  # Jupyter 中直接 await，脚本中用 asyncio.run(main())

## 2. 核心概念

| 概念 | 说明 |
|------|------|
| `coroutine` | 用 `async def` 定义的函数，返回一个协程对象 |
| `await` | 暂停当前协程，等待另一个协程完成，期间 event loop 可执行其他任务 |
| `asyncio.gather` | 并发运行多个协程，等待全部完成 |
| `asyncio.create_task` | 将协程包装为 Task，立即开始执行 |
| `event loop` | 单线程事件循环，统一调度所有协程 |

In [ ]:
# asyncio.create_task 示例：任务可以在后台运行
async def background_monitor():
    """模拟 Agent 监控任务，后台持续运行"""
    for i in range(3):
        await asyncio.sleep(0.5)
        print(f"  [监控] 检查点 {i+1}")

async def main_work():
    """主任务"""
    # 启动后台任务（不 await，让它在后台跑）
    monitor_task = asyncio.create_task(background_monitor())
    
    print("主任务开始执行...")
    await asyncio.sleep(1.2)  # 主任务执行
    print("主任务完成")
    
    await monitor_task  # 等待后台任务结束

await main_work()

## 3. 异常处理

In [ ]:
import random

async def flaky_llm_call(provider: str) -> str:
    """模拟可能失败的 LLM 调用"""
    await asyncio.sleep(0.5)
    if random.random() < 0.3:  # 30% 概率失败
        raise Exception(f"{provider} 调用超时")
    return f"{provider}: '什么是 AI Agent?'"

async def safe_llm_call(provider: str) -> str:
    """带错误处理的 LLM 调用包装"""
    try:
        return await flaky_llm_call(provider)
    except Exception as e:
        return f"{provider}: 调用失败 - {e}"

# return_exceptions=True: 不让一个失败影响其他任务
async def query_all_providers():
    results = await asyncio.gather(
        safe_llm_call("OpenAI"),
        safe_llm_call("Claude"),
        safe_llm_call("通义千问"),
    )
    for r in results:
        print(r)

await query_all_providers()

## 4. 实战：并发调用多个 LLM API

这是实际 Agent 开发中的常见模式——同时向多个模型提问，取最快或最好的结果。

In [ ]:
import sys
sys.path.insert(0, "..")
from utils.llm_client import call_llm_async

async def compare_llms(question: str):
    """同时向多个 LLM 提问，对比结果"""
    print(f"问题: {question}\n")
    
    tasks = [
        ("OpenAI gpt-4o-mini", call_llm_async(question, provider="openai", model="gpt-4o-mini")),
        ("Claude haiku",       call_llm_async(question, provider="anthropic", model="claude-haiku-4-5-20251001")),
    ]
    
    start = time.time()
    results = await asyncio.gather(
        *[task for _, task in tasks],
        return_exceptions=True
    )
    elapsed = time.time() - start
    
    for (name, _), result in zip(tasks, results):
        if isinstance(result, Exception):
            print(f"[{name}] 失败: {result}")
        else:
            print(f"[{name}]\n{result[:200]}...\n")
    
    print(f"并发总耗时: {elapsed:.2f}s")

# 如果配置了 API Keys 则运行
await compare_llms("用 3 句话解释什么是 AI Agent，以及它和普通 LLM 的区别")

## 5. asyncio.timeout：超时控制

In [ ]:
async def slow_tool(seconds: float = 5.0) -> str:
    await asyncio.sleep(seconds)
    return "工具执行完成"

async def call_with_timeout(timeout: float = 2.0):
    try:
        # Python 3.11+ 的超时语法
        async with asyncio.timeout(timeout):
            result = await slow_tool(5.0)
            print(result)
    except asyncio.TimeoutError:
        print(f"超时（>{timeout}s），使用降级方案")
        return "默认答案"

await call_with_timeout(2.0)

## 总结

| 场景 | 用法 |
|------|------|
| 并发多个 API 调用 | `asyncio.gather(task1, task2, ...)` |
| 后台持续运行任务 | `asyncio.create_task(coro)` |
| 带超时的调用 | `async with asyncio.timeout(seconds)` |
| 容错并发 | `gather(..., return_exceptions=True)` |

**自检**: 不看代码，能手写一个并发调用 3 个异步函数并收集结果的程序吗？

**下一节**: `02_decorators_generators.ipynb` — 装饰器与生成器